In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder as ohe
from sklearn.preprocessing import StandardScaler as ss
from sklearn.compose import ColumnTransformer as ct
from sklearn.svm import SVC as svc
from sklearn.tree import DecisionTreeClassifier as dtc
from sklearn.model_selection import train_test_split as tts
from sklearn.ensemble import BaggingClassifier as bc
from sklearn.model_selection import GridSearchCV as gsc

In [5]:
c=ct([("numeric",ss(),["Age","RestingBP","Cholesterol","MaxHR","Oldpeak"]),
      ("categorical",ohe(drop="first"),["Sex","ChestPainType","RestingECG","ExerciseAngina","ST_Slope"])],remainder="passthrough")

In [7]:
p1=Pipeline([("transformer",c),("model",svc())])
p2=Pipeline([("transformer",c),("model",dtc())])

In [8]:
x=df.drop(["HeartDisease"],axis=1)
y=df["HeartDisease"]

In [9]:
x_train,x_test,y_train,y_test=tts(x,y,test_size=0.2,random_state=18)

In [10]:
p1.fit(x_train,y_train)
p1.score(x_test,y_test)

0.8478260869565217

In [11]:
p2.fit(x_train,y_train)
p2.score(x_test,y_test)

0.7336956521739131

In [18]:
b1=bc(estimator=svc(),n_estimators=100,max_samples=0.8,oob_score=True,random_state=18)
b2=bc(estimator=dtc(),n_estimators=100,max_samples=0.8,oob_score=True,random_state=18)

In [21]:
p3=Pipeline([("transformer",c),("model",b1)])
p4=Pipeline([("transformer",c),("model",b2)])

In [22]:
p3.fit(x_train,y_train)
p3.score(x_test,y_test)

0.842391304347826

In [23]:
p4.fit(x_train,y_train)
p4.score(x_test,y_test)

0.8043478260869565

In [28]:
param_grid={"model__n_estimators":[25,50,75,100],"model__max_samples":[0.6,0.8,1.0]}

In [29]:
g1=gsc(p3,param_grid,cv=5)
g2=gsc(p4,param_grid,cv=5)

In [30]:
g1.fit(x_train,y_train)
g2.fit(x_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=18))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_samples': [0.6, 0.8, ...], 'model__n_estimators': [25, 50, ...]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbos

In [31]:
g1.best_score_

np.float64(0.8719597428012301)

In [32]:
g2.best_score_

np.float64(0.8542260739912402)

In [34]:
g1.best_params_

{'model__max_samples': 1.0, 'model__n_estimators': 100}

In [35]:
g2.best_params_

{'model__max_samples': 0.8, 'model__n_estimators': 100}

In [36]:
model1=g1.best_estimator_
model2=g2.best_estimator_

In [37]:
model1.score(x_test,y_test)

0.842391304347826

In [38]:
model2.score(x_test,y_test)

0.8043478260869565